# 05c - Deep Learning Forte: InceptionTime 1D

**Objetivo:** treinar uma arquitetura InceptionTime 1D mais robusta usando sinais brutos `records500`.

A escolha por InceptionTime 1D é adequada para séries temporais porque combina kernels de diferentes tamanhos, capturando padrões locais e mais longos no ECG sem usar Transformer. O teste permanece reservado para a avaliação final.

In [1]:
from pathlib import Path
import sys
import time
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tcc_ecg.config import load_config
from tcc_ecg.data import prepare_metadata
from tcc_ecg.deep_learning_strong import get_deep_learning_strong_config, train_deep_learning_strong
from tcc_ecg.paths import resolve_project_path
from tcc_ecg.utils import setup_logging

setup_logging()
config = load_config()
strong_config = get_deep_learning_strong_config(config)
config['data']['signal_frequency'] = int(strong_config['frequency'])
print('Modelo:', strong_config['model_name'])
print('Frequencia:', config['data']['signal_frequency'])
print('Epocas maximas:', strong_config['epochs'])
print('Batch size:', strong_config['batch_size'])
print('Loss:', strong_config['loss_type'])

Modelo: inceptiontime1d_strong
Frequencia: 500
Epocas maximas: 80
Batch size: 64
Loss: focal


## Treinamento

A normalização por canal é calculada somente no treino. As aumentações são leves e aplicadas apenas ao treino. O checkpoint é escolhido por `val_f1_macro`, sem uso do teste durante o tuning.

In [2]:
start = time.time()
metrics_path = resolve_project_path('reports/tables/deep_learning_strong_metrics.csv', config['project_root'])
checkpoint_path = resolve_project_path('models/deep_learning_strong_best.pt', config['project_root'])
if metrics_path.exists() and checkpoint_path.exists():
    metrics = pd.read_csv(metrics_path)
    elapsed = 0.0
    print('Artefatos existentes encontrados; treino nao foi reexecutado.')
    print('Checkpoint:', checkpoint_path)
else:
    metadata = prepare_metadata(config, save_summary=False)
    results = train_deep_learning_strong(metadata, config)
    elapsed = time.time() - start
    metrics = pd.DataFrame([results['metrics']])
    print('Checkpoint:', results['checkpoint'])
    print('Device:', results['device'])
display(metrics)
print('Tempo total notebook (s):', round(elapsed, 2))

Artefatos existentes encontrados; treino nao foi reexecutado.
Checkpoint: C:\dev\personal\ptb-xl\models\deep_learning_strong_best.pt


,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,model,split,signal_frequency,smote,device,best_epoch,epochs_trained,training_seconds
0,0.782424,0.680584,0.705686,0.680584,0.684006,0.780069,inceptiontime1d_strong,test,500,False,cpu,12,30,27118.132024


Tempo total notebook (s): 0.0
